# LLM Sandbox: Summarize Cards  
__Objective:__ Fine-tune an LLM to summarize cards based on their function as defined by the scryfall tags.

## Packages and Data

In [1]:
# packages

## project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## from project directory
from src.data_gathering.scryfall import Scryfall
from src.data_gathering.scryfall_tags import ScryfallTags

## general
import json

In [2]:
# params
from src.config import TOTAL_CARDS, RATE_LIMIT_SECONDS, MAX_LOAD_TIME
from src.config import SAVE_EVERY, OUTPUT_PATH
print(f'Total Cards = {TOTAL_CARDS}')

Total Cards = 200


In [3]:
# read scryfall data
sf = Scryfall()
sf.read_data()

Scryfall Cards
	Source = ../data/oracle-cards.json
	Card Count = 36680
	Read On = 2026-02-03


In [4]:
for k, v in sf.data[0].items():
    print(f'{k}: {v}')

object: card
id: a471b306-4941-4e46-a0cb-d92895c16f8a
oracle_id: 00037840-6089-42ec-8c5c-281f9f474504
multiverse_ids: [692174]
mtgo_id: 137223
tcgplayer_id: 615195
cardmarket_id: 807933
name: Nissa, Worldsoul Speaker
lang: en
released_at: 2025-02-14
uri: https://api.scryfall.com/cards/a471b306-4941-4e46-a0cb-d92895c16f8a
scryfall_uri: https://scryfall.com/card/drc/13/nissa-worldsoul-speaker?utm_source=api
layout: normal
highres_image: True
image_status: highres_scan
image_uris: {'small': 'https://cards.scryfall.io/small/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a.jpg?1738355341', 'normal': 'https://cards.scryfall.io/normal/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a.jpg?1738355341', 'large': 'https://cards.scryfall.io/large/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a.jpg?1738355341', 'png': 'https://cards.scryfall.io/png/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a.png?1738355341', 'art_crop': 'https://cards.scryfall.io/art_crop/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a

In [5]:
# out = []
# for card in sf.data[:10000]:
#     t = card['type_line'].split(' — ', maxsplit = 1)
#     out.append(t[0])
# set(out)

__NOTE:__ The scryfall API data _does not_ natively include scryfall tags. So we need to scrape the web to retrieve this data.

In [6]:
per_card_time = {
    3:   ((7 * 60) + 14) / TOTAL_CARDS,
    2.5: ((6 * 54) + 14) / TOTAL_CARDS,
    2.25: ((6 * 60) + 6) / TOTAL_CARDS,
    2:   ((5 * 60) + 34) / TOTAL_CARDS
}

for k, v in per_card_time.items():
    total_seconds = v * len(sf.data)
    total_hours = total_seconds / (60 * 60)

    print(
        f'Projected Hours To Scrape Tags w/ {k}s Wait: {total_hours:.2f}h '
        f'({v}sec per card)'
    )

Projected Hours To Scrape Tags w/ 3s Wait: 22.11h (2.17sec per card)
Projected Hours To Scrape Tags w/ 2.5s Wait: 17.22h (1.69sec per card)
Projected Hours To Scrape Tags w/ 2.25s Wait: 18.65h (1.83sec per card)
Projected Hours To Scrape Tags w/ 2s Wait: 17.02h (1.67sec per card)


In [7]:
# scrape tagger.scryfall.com for each cards tags
# NOTE: This may be a really long run time.
tags = ScryfallTags()
tags.scrape_all_cards(
    data = sf.data,
    total_cards = TOTAL_CARDS,
    rate_limit_seconds = RATE_LIMIT_SECONDS,
    max_load_time = MAX_LOAD_TIME,
    save_every = SAVE_EVERY,
    output_path = OUTPUT_PATH
)
tags.data

199it [13:30,  4.07s/it]

Processed 199 cards Scryfall tags.
Data successfully saved to ../reports/scryfall_tags.json


[{'00037840-6089-42ec-8c5c-281f9f474504': ['cost ignorer',
   'counter fuel-energy',
   'energy generator',
   'free-cast-another',
   'landfall',
   'triggered ability']},
 {'000492bf-7eaa-4939-a51c-4eef74e4c1d1': ['burn creature',
   'conjure-creature',
   'conjure-named',
   'conjure-to-library',
   'free-cast-another',
   'spot removal',
   'triggered ability',
   'virtual french vanilla']},
 {'0004ebd0-dfd6-4276-b4a6-de0003e94237': ['Winter Orb',
   'deprecated untapped artifact',
   'mass land denial',
   'portmanteau\nAnnotation: It combines the effects of Stasis and Winter Orb, and its name is loosely a portmanteau of the two.',
   'stasis',
   'symmetrical',
   'synergy-vigilance']},
 {'0006faf6-7a61-426c-9034-579f2cfcfa83': ['Cryoshatter',
   'Shattered Ego',
   'shrink']},
 {'00078ea3-0462-4a6e-b7b1-25fea012b2b7': ['animate artifact',
   'artifactify',
   'delayed trigger',
   'gives haste',
   'sacrifice outlet-artifact',
   'sacrifice outlet-creature',
   'shapechange',
  